# Health Insurance Premium Prediction - Model Training

Run every cell top to bottom in Google Colab. At the end, `model.pkl` and `metrics.json` are downloaded.
Put both files in the root of your GitHub project (next to `app.py`).

## 1. Load the data

In [ ]:
import pandas as pd, numpy as np, json, sklearn, joblib

URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)
print("scikit-learn version:", sklearn.__version__)
print(df.shape)
df.head()

## 2. Quick exploration

In [ ]:
print(df.isna().sum())
print(df.describe())
df.groupby("smoker")["charges"].mean()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df["charges"].hist(bins=40, ax=ax[0]); ax[0].set_title("Charges distribution")
df.boxplot(column="charges", by="smoker", ax=ax[1]); ax[1].set_title("Charges by smoker")
ax[2].scatter(df["bmi"], df["charges"], c=(df["smoker"] == "yes"), cmap="coolwarm", s=10)
ax[2].set_title("BMI vs charges (red = smoker)")
plt.show()

## 3. Build and compare models

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

FEATURES = ["age", "sex", "bmi", "children", "smoker", "region"]   # same order as app.py
X, y = df[FEATURES], df["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def make_pipe(estimator):
    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), ["sex", "smoker", "region"])],
        remainder="passthrough")
    return Pipeline([("pre", pre), ("model", estimator)])

candidates = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}
scores = {}
for name, est in candidates.items():
    pipe = make_pipe(est)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2").mean()
    scores[name] = cv
    print(f"{name:20s} CV R2 = {cv:.3f}")
best_name = max(scores, key=scores.get)
print("Best:", best_name)

## 4. Train the best model and evaluate

In [ ]:
best = make_pipe(candidates[best_name])
best.fit(X_train, y_train)
pred = best.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
print(f"R2={r2:.3f}  MAE={mae:,.0f}  RMSE={rmse:,.0f}")

plt.scatter(y_test, pred, s=10); plt.plot([0, y.max()], [0, y.max()], "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title(best_name); plt.show()

## 5. Feature importance (for the dashboard)

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(best, X_test, y_test, n_repeats=10, random_state=42)
importances = {f: round(float(v), 4) for f, v in sorted(zip(FEATURES, perm.importances_mean), key=lambda t: -t[1])}
importances

## 6. Save and download

In [ ]:
# Retrain on ALL data for the final model
final = make_pipe(candidates[best_name]).fit(X, y)
joblib.dump(final, "model.pkl")

metrics = {"model": best_name, "r2": round(r2, 3), "mae": round(mae, 0), "rmse": round(rmse, 0),
           "importances": importances, "rows": int(len(df)), "sklearn_version": sklearn.__version__}
json.dump(metrics, open("metrics.json", "w"), indent=2)
print(metrics)

from google.colab import files
files.download("model.pkl")
files.download("metrics.json")

## 7. Important
Note the **scikit-learn version** printed in step 1. In your GitHub project's `requirements.txt`, set
`scikit-learn==<that version>` so the deployed app can read `model.pkl`.